In [1]:
from pyspark.sql import SparkSession

# POINT D'ENTREE = Création d'une SparkSession locale
spark = SparkSession.builder.appName("TutoDataFrame_PySpark").master("local[*]").getOrCreate()

print(spark)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/20 16:26:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


***Charger et lire les 3 types de fichiers***


In [2]:
import os

base_path="data-68ed/data/input/"

orders_df = spark.read.option("multiline", "true").json(os.path.join(base_path, "orders_2025-03-01.json"))
customers_df = spark.read.csv(os.path.join(base_path, "customers.csv"), header=True, inferSchema=True)
refunds_df = spark.read.csv(os.path.join(base_path, "refunds.csv"), header=True, inferSchema=True)

refunds_df.show(5)
orders_df.show(5)
customers_df.show(5)

+---------+-------------+------+----------+-------------------+
|refund_id|     order_id|amount|    reason|         created_at|
+---------+-------------+------+----------+-------------------+
|  R000001|O202503010089| -6.68|     delay|2025-03-01 14:03:41|
|  R000002|O202503010038| -8.89|   gesture|2025-03-01 22:16:56|
|  R000003|O202503010008|-15.23|item_issue|2025-03-01 20:06:25|
|  R000004|O202503010073| -2.47|    coupon|2025-03-01 20:02:46|
|  R000005|O202503010005| -3.83|   gesture|2025-03-01 09:58:15|
+---------+-------------+------+----------+-------------------+
only showing top 5 rows

+-------+-------------------+-----------+-------------------+-------------+--------------+
|channel|         created_at|customer_id|              items|     order_id|payment_status|
+-------+-------------------+-----------+-------------------+-------------+--------------+
|    app|2025-03-01 20:36:44|      C0793|[{4, SKU001, 24.9}]|O202503010001|          paid|
|    web|2025-03-01 11:30:49|      

**nettoyer les données et créer les colonnes**

--1-- Extraire la date du champs 'created_at' dans orders_df et refunds_df et la mettre dans un champs "date"

In [3]:
from pyspark.sql.functions import to_date, col

orders_df = orders_df.withColumn("date", to_date(col("created_at")))
refunds_df = refunds_df.withColumn("date", to_date(col("created_at")))

orders_df.show(5)
refunds_df.show(5)

orders_df.printSchema()


+-------+-------------------+-----------+-------------------+-------------+--------------+----------+
|channel|         created_at|customer_id|              items|     order_id|payment_status|      date|
+-------+-------------------+-----------+-------------------+-------------+--------------+----------+
|    app|2025-03-01 20:36:44|      C0793|[{4, SKU001, 24.9}]|O202503010001|          paid|2025-03-01|
|    web|2025-03-01 11:30:49|      C0676| [{4, SKU042, 7.5}]|O202503010002|          paid|2025-03-01|
|    web|2025-03-01 07:27:00|      C0642| [{1, SKU014, 5.0}]|O202503010003|          paid|2025-03-01|
|    web|2025-03-01 14:28:46|      C0283| [{2, SKU024, 4.0}]|O202503010004|       pending|2025-03-01|
|    web|2025-03-01 22:29:42|      C0571| [{1, SKU001, 2.5}]|O202503010005|          paid|2025-03-01|
+-------+-------------------+-----------+-------------------+-------------+--------------+----------+
only showing top 5 rows

+---------+-------------+------+----------+--------------

--2-- Aggréger les ventes par ville et canal

In [4]:
#**ajuster agrégation en spark: extraire nb d'articles vendus et total par ligne

from pyspark.sql.functions import col, explode, sum as spark_sum

#joindre "city" à orders sur la clé "customer_id"
orders_base = orders_df.join(customers_df.select("customer_id", "city"), on="customer_id", how="left")

# Exploser items pr calculer les ventes 
orders_exploded = orders_base.withColumn("item", explode(col("items"))) \
                             .withColumn("items_sold", col("item.qty")) \
                             .withColumn("total_amount", col("item.qty") * col("item.unit_price"))

orders_exploded.show(5)
orders_base.show(5)

+-----------+-------+-------------------+-------------------+-------------+--------------+----------+---------+-----------------+----------+------------+
|customer_id|channel|         created_at|              items|     order_id|payment_status|      date|     city|             item|items_sold|total_amount|
+-----------+-------+-------------------+-------------------+-------------+--------------+----------+---------+-----------------+----------+------------+
|      C0793|    app|2025-03-01 20:36:44|[{4, SKU001, 24.9}]|O202503010001|          paid|2025-03-01| Toulouse|{4, SKU001, 24.9}|         4|        99.6|
|      C0676|    web|2025-03-01 11:30:49| [{4, SKU042, 7.5}]|O202503010002|          paid|2025-03-01|Marseille| {4, SKU042, 7.5}|         4|        30.0|
|      C0642|    web|2025-03-01 07:27:00| [{1, SKU014, 5.0}]|O202503010003|          paid|2025-03-01| Toulouse| {1, SKU014, 5.0}|         1|         5.0|
|      C0283|    web|2025-03-01 14:28:46| [{2, SKU024, 4.0}]|O202503010004| 

In [5]:
#on separe les deux types d'agrégations: pour orders_count et unique_cutomers, on utilise order_base qui contient 1 ligne/commande

from pyspark.sql.functions import sum as spark_sum, count, approx_count_distinct

orders_agg = (
    orders_base.groupBy("date", "city", "channel")
    .agg(
        count("order_id").alias("orders_count"),
        approx_count_distinct("customer_id").alias("unique_customers")
    )
)


# pour items_sold et gross_revenue_eur, utiliser orders_exploded (1ligne/item)
sales_items_agg = (
    orders_exploded.groupBy("date", "city", "channel")
    .agg(
        spark_sum("items_sold").alias("items_sold"),
        spark_sum("total_amount").alias("gross_revenue_eur")
    )
)


In [6]:
#count('order_id'): compte ttes les commandes ; approx_count_distinct: compte les clients uniques
#.reset_index inutile: spark produit un df plat

sales_agg = orders_agg.join(
    sales_items_agg,
    on=["date", "city", "channel"],
    how="left"
)
sales_agg.show(5)
sales_agg.printSchema()



25/11/20 16:27:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+---------+-------+------------+----------------+----------+-----------------+
|      date|     city|channel|orders_count|unique_customers|items_sold|gross_revenue_eur|
+----------+---------+-------+------------+----------------+----------+-----------------+
|2025-03-01| Bordeaux|    app|           8|               7|        62|806.5999999999999|
|2025-03-01|    Paris|    web|           9|               8|        61|            822.8|
|2025-03-01|   Nantes|    web|           2|               2|         9|             84.9|
|2025-03-01|Marseille|    app|          11|              11|        84|            768.3|
|2025-03-01|     Nice|    app|           7|               7|        56|813.3999999999999|
+----------+---------+-------+------------+----------------+----------+-----------------+
only showing top 5 rows

root
 |-- date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- orders_count: long (nullable = false)
 |-- uni

--3-- Enrichir les remboursements avec city et channel

In [7]:
#table de ref : order_enriched qui va contenir city

refunds_enriched = refunds_df.join(
    orders_base.select("order_id", "city", "channel"),
    on="order_id",
    how="left")

#agregation
refunds_agg = (refunds_enriched.groupBy("date","channel","city").agg(
    spark_sum("amount").alias("refunds_eur")))

#on vérifie
refunds_agg.show(10)
refunds_agg.printSchema()


+----------+-------+---------+-------------------+
|      date|channel|     city|        refunds_eur|
+----------+-------+---------+-------------------+
|2025-03-01|    app|    Lille|             -24.47|
|2025-03-23|   NULL|     NULL|            -569.86|
|2025-03-16|   NULL|     NULL| -282.7300000000001|
|2025-03-03|   NULL|     NULL|-373.21000000000004|
|2025-03-08|   NULL|     NULL|-507.72999999999996|
|2025-03-12|   NULL|     NULL| -523.1699999999998|
|2025-03-07|   NULL|     NULL|-252.38999999999996|
|2025-03-13|   NULL|     NULL|            -287.25|
|2025-03-01|    web|Marseille|              -6.05|
|2025-03-31|   NULL|     NULL|            -245.98|
+----------+-------+---------+-------------------+
only showing top 10 rows

root
 |-- date: date (nullable = true)
 |-- channel: string (nullable = true)
 |-- city: string (nullable = true)
 |-- refunds_eur: double (nullable = true)



--3-- combiner ventes +remboursmeents

In [8]:
#on fusionne sales_agg et refunds_agg sur le trio clé 'date', 'city', 'channel' en gardant toutes les ventes (left join), les NaN sont remplacés par 0

final_summary = sales_agg.join(
    refunds_agg,
    on=["date", "city", "channel"],
    how="left"
).fillna(0, subset=["refunds_eur"])


--4-- on calcule le revenu net (CA - refund) et on standardise résultat à 2 décimales

In [9]:
from pyspark.sql.functions import round

final_summary = final_summary.withColumn(
    "net_revenue_eur",
    col("gross_revenue_eur") + col("refunds_eur")
)

final_summary = final_summary.withColumn("gross_revenue_eur", round(col("gross_revenue_eur"), 2)) \
                             .withColumn("refunds_eur", round(col("refunds_eur"), 2)) \
                             .withColumn("net_revenue_eur", round(col("net_revenue_eur"), 2))

final_summary.show(5)


+----------+---------+-------+------------+----------------+----------+-----------------+-----------+---------------+
|      date|     city|channel|orders_count|unique_customers|items_sold|gross_revenue_eur|refunds_eur|net_revenue_eur|
+----------+---------+-------+------------+----------------+----------+-----------------+-----------+---------------+
|2025-03-01| Bordeaux|    app|           8|               7|        62|            806.6|     -50.91|         755.69|
|2025-03-01|    Paris|    web|           9|               8|        61|            822.8|     -13.53|         809.27|
|2025-03-01|   Nantes|    web|           2|               2|         9|             84.9|        0.0|           84.9|
|2025-03-01|Marseille|    app|          11|              11|        84|            768.3|     -69.13|         699.17|
|2025-03-01|     Nice|    app|           7|               7|        56|            813.4|      -24.7|          788.7|
+----------+---------+-------+------------+-------------

--5-- Réorganiser les colonnes

In [10]:
final_summary = final_summary.select(
    "date",
    "city",
    "channel",
    "orders_count",
    "unique_customers",
    "items_sold",
    "gross_revenue_eur",
    "refunds_eur",
    "net_revenue_eur"
)

final_summary.show(5)


+----------+---------+-------+------------+----------------+----------+-----------------+-----------+---------------+
|      date|     city|channel|orders_count|unique_customers|items_sold|gross_revenue_eur|refunds_eur|net_revenue_eur|
+----------+---------+-------+------------+----------------+----------+-----------------+-----------+---------------+
|2025-03-01| Bordeaux|    app|           8|               7|        62|            806.6|     -50.91|         755.69|
|2025-03-01|    Paris|    web|           9|               8|        61|            822.8|     -13.53|         809.27|
|2025-03-01|   Nantes|    web|           2|               2|         9|             84.9|        0.0|           84.9|
|2025-03-01|Marseille|    app|          11|              11|        84|            768.3|     -69.13|         699.17|
|2025-03-01|     Nice|    app|           7|               7|        56|            813.4|      -24.7|          788.7|
+----------+---------+-------+------------+-------------

In [11]:
##tester cohérence entre deux fichiers
import pandas as pd
import os

csv_path = "daily_summary/daily_summary_20250301.csv"
daily_summary_pd = pd.read_csv(csv_path, sep=';')

daily_summary_pd.head(5)
daily_summary_pd.dtypes


date                  object
city                  object
channel               object
orders_count           int64
unique_customers       int64
items_sold             int64
gross_revenue_eur    float64
refunds_eur          float64
net_revenue_eur      float64
dtype: object

In [12]:
spark_df_pd = final_summary.toPandas()
spark_df_pd.head(5)


,date,city,channel,orders_count,unique_customers,items_sold,gross_revenue_eur,refunds_eur,net_revenue_eur
0,2025-03-01,Bordeaux,app,8,7,62,806.6,-50.91,755.69
1,2025-03-01,Paris,web,9,8,61,822.8,-13.53,809.27
2,2025-03-01,Nantes,web,2,2,9,84.9,0.00,84.90
3,2025-03-01,Marseille,app,11,11,84,768.3,-69.13,699.17
4,2025-03-01,Nice,app,7,7,56,813.4,-24.70,788.70


In [13]:
print("Total gross revenue (CSV):", daily_summary_pd["gross_revenue_eur"].sum())
print("Total gross revenue (Spark):", spark_df_pd["gross_revenue_eur"].sum())

print("Total refunds (CSV):", daily_summary_pd["refunds_eur"].sum())
print("Total refunds (Spark):", spark_df_pd["refunds_eur"].sum())

print("Total net revenue (CSV):", daily_summary_pd["net_revenue_eur"].sum())
print("Total net revenue (Spark):", spark_df_pd["net_revenue_eur"].sum())


Total gross revenue (CSV): 6365.7
Total gross revenue (Spark): 8225.3
Total refunds (CSV): -321.96000000000004
Total refunds (Spark): -384.07
Total net revenue (CSV): 6043.74
Total net revenue (Spark): 7841.2300000000005


In [14]:
import numpy as np

# Tri pour s'assurer que les lignes correspondent
daily_summary_pd_sorted = daily_summary_pd.sort_values(["date","city","channel"]).reset_index(drop=True)
spark_df_pd_sorted = spark_df_pd.sort_values(["date","city","channel"]).reset_index(drop=True)

comparison = np.isclose(
    spark_df_pd_sorted[["orders_count","unique_customers","items_sold","gross_revenue_eur","refunds_eur","net_revenue_eur"]],
    daily_summary_pd_sorted[["orders_count","unique_customers","items_sold","gross_revenue_eur","refunds_eur","net_revenue_eur"]],
    rtol=1e-5
)

print("Toutes les lignes et colonnes correspondent :", comparison.all())


Toutes les lignes et colonnes correspondent : False


In [15]:
mask = ~comparison.all(axis=1)
daily_summary_pd_sorted[mask]
spark_df_pd_sorted[mask]


,date,city,channel,orders_count,unique_customers,items_sold,gross_revenue_eur,refunds_eur,net_revenue_eur
0,2025-03-01,Bordeaux,app,8,7,62,806.6,-50.91,755.69
3,2025-03-01,Lille,web,7,6,55,562.2,-14.23,547.97
4,2025-03-01,Lyon,app,8,7,62,730.3,-10.86,719.44
5,2025-03-01,Lyon,web,6,6,34,440.1,-3.85,436.25
6,2025-03-01,Marseille,app,11,11,84,768.3,-69.13,699.17
7,2025-03-01,Marseille,web,9,8,69,611.8,-6.05,605.75
9,2025-03-01,Nantes,web,2,2,9,84.9,0.00,84.90
10,2025-03-01,Nice,app,7,7,56,813.4,-24.70,788.70
11,2025-03-01,Nice,web,4,4,34,285.1,-26.84,258.26
12,2025-03-01,Paris,app,5,5,32,508.0,-33.89,474.11


In [16]:
import numpy as np

# Tri pour aligner les lignes
daily_summary_pd_sorted = daily_summary_pd.sort_values(["date","city","channel"]).reset_index(drop=True)
spark_df_pd_sorted = spark_df_pd.sort_values(["date","city","channel"]).reset_index(drop=True)

# Comparaison ligne par ligne
comparison = np.isclose(
    spark_df_pd_sorted[["orders_count","unique_customers","items_sold","gross_revenue_eur","refunds_eur","net_revenue_eur"]],
    daily_summary_pd_sorted[["orders_count","unique_customers","items_sold","gross_revenue_eur","refunds_eur","net_revenue_eur"]],
    rtol=1e-5
)

# Lignes différentes
diff_mask = ~comparison.all(axis=1)
diff_rows_spark = spark_df_pd_sorted[diff_mask]
diff_rows_csv = daily_summary_pd_sorted[diff_mask]

print("Nombre de lignes différentes :", diff_mask.sum())
diff_rows_spark
diff_rows_csv


Nombre de lignes différentes : 13


,date,city,channel,orders_count,unique_customers,items_sold,gross_revenue_eur,refunds_eur,net_revenue_eur
0,2025-03-01,Bordeaux,app,4,4,27,325.4,-25.55,299.85
3,2025-03-01,Lille,web,3,3,37,417.0,-14.23,402.77
4,2025-03-01,Lyon,app,7,7,59,707.8,-10.86,696.94
5,2025-03-01,Lyon,web,3,3,21,257.9,-2.81,255.09
6,2025-03-01,Marseille,app,9,9,69,594.4,-62.45,531.95
7,2025-03-01,Marseille,web,8,7,55,497.7,-6.05,491.65
9,2025-03-01,Nantes,web,1,1,8,77.4,0.00,77.40
10,2025-03-01,Nice,app,6,6,50,733.6,-24.70,708.90
11,2025-03-01,Nice,web,3,3,23,173.2,-23.62,149.58
12,2025-03-01,Paris,app,3,3,26,423.3,-21.80,401.50


In [17]:
# Arrondir les colonnes financières pour comparaison stricte
cols_fin = ["gross_revenue_eur", "refunds_eur", "net_revenue_eur"]
spark_df_pd_sorted[cols_fin] = spark_df_pd_sorted[cols_fin].round(2)
daily_summary_pd_sorted[cols_fin] = daily_summary_pd_sorted[cols_fin].round(2)


In [18]:
comparison_rounded = np.isclose(
    spark_df_pd_sorted[cols_fin],
    daily_summary_pd_sorted[cols_fin],
    rtol=1e-5
)

diff_mask_rounded = ~comparison_rounded.all(axis=1)
print("Lignes différentes après arrondi :", diff_mask_rounded.sum())


Lignes différentes après arrondi : 13


In [19]:
# Tri pour aligner les lignes
daily_summary_pd_sorted = daily_summary_pd.sort_values(["date","city","channel"]).reset_index(drop=True)
spark_df_pd_sorted = spark_df_pd.sort_values(["date","city","channel"]).reset_index(drop=True)

# Colonnes numériques à comparer
cols_num = ["orders_count","unique_customers","items_sold","gross_revenue_eur","refunds_eur","net_revenue_eur"]

# Comparaison arrondie
comparison = np.isclose(
    spark_df_pd_sorted[cols_num].round(2),
    daily_summary_pd_sorted[cols_num].round(2),
    rtol=1e-5
)

# Masque des lignes différentes
diff_mask = ~comparison.all(axis=1)

# Lignes différentes
diff_rows_spark = spark_df_pd_sorted[diff_mask]
diff_rows_csv = daily_summary_pd_sorted[diff_mask]

print("Nombre de lignes différentes :", diff_mask.sum())


Nombre de lignes différentes : 13


In [20]:
for col in cols_num:
    diff_col_mask = spark_df_pd_sorted[col].round(2) != daily_summary_pd_sorted[col].round(2)
    if diff_col_mask.any():
        print(f"\nDifférences dans la colonne '{col}':")
        print(pd.concat([
            spark_df_pd_sorted.loc[diff_col_mask, ["date","city","channel", col]].rename(columns={col: col+"_spark"}),
            daily_summary_pd_sorted.loc[diff_col_mask, [col]].rename(columns={col: col+"_csv"})
        ], axis=1))



Différences dans la colonne 'orders_count':
          date       city channel  orders_count_spark  orders_count_csv
0   2025-03-01   Bordeaux     app                   8                 4
3   2025-03-01      Lille     web                   7                 3
4   2025-03-01       Lyon     app                   8                 7
5   2025-03-01       Lyon     web                   6                 3
6   2025-03-01  Marseille     app                  11                 9
7   2025-03-01  Marseille     web                   9                 8
9   2025-03-01     Nantes     web                   2                 1
10  2025-03-01       Nice     app                   7                 6
11  2025-03-01       Nice     web                   4                 3
12  2025-03-01      Paris     app                   5                 3
13  2025-03-01      Paris     web                   9                 7
15  2025-03-01   Toulouse     web                   9                 6

Différences dans l

In [24]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, explode, sum as _sum, count as _count, countDistinct, round as _round

# --- 1️⃣ Initialiser Spark ---
spark = SparkSession.builder \
    .appName("DailySalesDebug") \
    .master("local[*]") \
    .getOrCreate()

# --- 2️⃣ Chemins ---
base_input_path = os.path.join("data-68ed", "data", "input")

# --- 3️⃣ Charger les données ---
customers_df = spark.read.csv(
    os.path.join(base_input_path, "customers.csv"),
    header=True,
    inferSchema=True
)

refunds_df = spark.read.csv(
    os.path.join(base_input_path, "refunds.csv"),
    header=True,
    inferSchema=True
)

orders_df = spark.read.option("multiline", "true").json(
    os.path.join(base_input_path, "orders_*.json")
)

# --- 4️⃣ Filtrer les clients actifs ---
active_customers_df = customers_df.filter(col("is_active") == True).select("customer_id", "city")

# --- 5️⃣ Filtrer les commandes payées ---
paid_orders_df = orders_df.filter(col("payment_status") == "paid")

# --- 6️⃣ Jointure avec clients actifs ---
orders_base = paid_orders_df.join(active_customers_df, "customer_id", "inner")

# --- 7️⃣ Ajouter une colonne date ---
orders_base = orders_base.withColumn("date", to_date(col("created_at")))
refunds_df = refunds_df.withColumn("date", to_date(col("created_at")))

# --- 8️⃣ Exploser les items et filtrer prix > 0 ---
orders_exploded = orders_base.withColumn("item", explode(col("items"))) \
                             .filter(col("item.unit_price") > 0) \
                             .withColumn("items_sold", col("item.qty")) \
                             .withColumn("total_amount", col("item.qty") * col("item.unit_price"))

# --- 9️⃣ Déduplication sur order_id si nécessaire ---
orders_base = orders_base.dropDuplicates(["order_id"])

# --- 🔟 Agrégation ventes ---
# a) orders_count et unique_customers (1 ligne/commande)
orders_agg = orders_base.groupBy("date", "city", "channel").agg(
    _count("order_id").alias("orders_count"),
    countDistinct("customer_id").alias("unique_customers")
)

# b) items_sold et gross_revenue_eur (1 ligne/item)
sales_items_agg = orders_exploded.groupBy("date", "city", "channel").agg(
    _sum("items_sold").alias("items_sold"),
    _sum("total_amount").alias("gross_revenue_eur")
)

# c) Fusion des deux agrégations
sales_agg = orders_agg.join(sales_items_agg, ["date", "city", "channel"], "left")

# --- 1️⃣1️⃣ Agrégation remboursements ---
refunds_enriched = refunds_df.join(
    orders_base.select("order_id", "city", "channel"), "order_id", "left"
)
refunds_agg = refunds_enriched.groupBy("date", "city", "channel").agg(
    _sum("amount").alias("refunds_eur")
)

# --- 1️⃣2️⃣ Combinaison ventes + remboursements ---
final_summary_df = sales_agg.join(refunds_agg, ["date", "city", "channel"], "left") \
                            .na.fill(0, ["refunds_eur"])

# --- 1️⃣3️⃣ Calcul net_revenue et arrondis ---
final_summary_df = final_summary_df.withColumn(
    "net_revenue_eur", col("gross_revenue_eur") + col("refunds_eur")
).withColumn("gross_revenue_eur", _round("gross_revenue_eur", 2)) \
 .withColumn("refunds_eur", _round("refunds_eur", 2)) \
 .withColumn("net_revenue_eur", _round("net_revenue_eur", 2))

# --- 1️⃣4️⃣ Colonnes dans l'ordre final ---
final_summary_df = final_summary_df.select(
    "date", "city", "channel", "orders_count", "unique_customers",
    "items_sold", "gross_revenue_eur", "refunds_eur", "net_revenue_eur"
)

# --- 1️⃣5️⃣ Affichage pour vérification ---
final_summary_df.show(50, truncate=False)


25/11/20 16:45:34 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
                                                                                

+----------+---------+-------+------------+----------------+----------+-----------------+-----------+---------------+
|date      |city     |channel|orders_count|unique_customers|items_sold|gross_revenue_eur|refunds_eur|net_revenue_eur|
+----------+---------+-------+------------+----------------+----------+-----------------+-----------+---------------+
|2025-03-05|Nice     |app    |5           |5               |32        |410.2            |0.0        |410.2          |
|2025-03-19|Bordeaux |web    |8           |8               |54        |698.4            |-28.96     |669.44         |
|2025-03-23|Nice     |app    |4           |4               |34        |306.5            |-45.27     |261.23         |
|2025-03-18|Paris    |app    |7           |7               |50        |499.6            |-38.58     |461.02         |
|2025-03-24|Toulouse |web    |4           |3               |17        |213.0            |-35.29     |177.71         |
|2025-03-11|Lyon     |app    |8           |8            

In [27]:
import pandas as pd

# Charger le CSV correctement
pandas_summary = pd.read_csv("daily_summary/daily_summary_20250301.csv", sep=';', index_col=0)

# S'assurer que 'date' est de type datetime
pandas_summary['date'] = pd.to_datetime(pandas_summary['date'])

# Vérifier les colonnes
print(pandas_summary.head())

# Convertir Spark DataFrame en Pandas pour la même date
spark_result_pd = final_summary_df.filter(col("date") == "2025-03-01") \
                                  .select("date", "city", "channel", "orders_count",
                                          "unique_customers", "items_sold",
                                          "gross_revenue_eur", "refunds_eur", "net_revenue_eur") \
                                  .orderBy("city", "channel") \
                                  .toPandas()

# S'assurer que 'date' est datetime pour Spark aussi
spark_result_pd['date'] = pd.to_datetime(spark_result_pd['date'])

# Comparer
comparison = spark_result_pd.merge(
    pandas_summary,
    on=["date", "city", "channel"],
    how="outer",
    suffixes=("_spark", "_pandas"),
    indicator=True
)

# Lignes différentes
diff_rows = comparison[comparison["_merge"] != "both"]
print(f"Lignes différentes ou absentes : {len(diff_rows)}")
display(diff_rows)


KeyError: 'date'

In [28]:
import pandas as pd

pandas_summary = pd.read_csv("daily_summary/daily_summary_20250301.csv", sep=';')
print(pandas_summary.columns)
print(pandas_summary.head())


Index(['date', 'city', 'channel', 'orders_count', 'unique_customers',
       'items_sold', 'gross_revenue_eur', 'refunds_eur', 'net_revenue_eur'],
      dtype='object')
         date      city channel  orders_count  unique_customers  items_sold  \
0  2025-03-01  Bordeaux     app             4                 4          27   
1  2025-03-01  Bordeaux     web             6                 6          25   
2  2025-03-01     Lille     app             4                 4          26   
3  2025-03-01     Lille     web             3                 3          37   
4  2025-03-01      Lyon     app             7                 7          59   

   gross_revenue_eur  refunds_eur  net_revenue_eur  
0              325.4       -25.55           299.85  
1              332.5       -39.73           292.77  
2              321.8       -24.47           297.33  
3              417.0       -14.23           402.77  
4              707.8       -10.86           696.94  


In [1]:
# --- 1️⃣ Charger le CSV Pandas ---
import pandas as pd

pandas_summary = pd.read_csv("daily_summary/daily_summary_20250301.csv", sep=';')
# S'assurer que la colonne date est de type datetime
pandas_summary['date'] = pd.to_datetime(pandas_summary['date'])
pandas_summary.head()

# --- 2️⃣ Récupérer le résultat Spark pour la même date ---
# Ici final_summary_df est le Spark DataFrame issu de ton script Spark
spark_result = final_summary_df.filter(col("date") == "2025-03-01") \
                               .select("date", "city", "channel", "orders_count",
                                       "unique_customers", "items_sold",
                                       "gross_revenue_eur", "refunds_eur", "net_revenue_eur") \
                               .orderBy("city", "channel")

# Convertir en Pandas pour comparaison
spark_summary_pd = spark_result.toPandas()

# S'assurer que les colonnes datetime sont compatibles
spark_summary_pd['date'] = pd.to_datetime(spark_summary_pd['date'])

# --- 3️⃣ Fusionner pour comparaison ---
comparison = spark_summary_pd.merge(
    pandas_summary,
    on=["date", "city", "channel"],
    how="outer",
    suffixes=("_spark", "_pandas"),
    indicator=True
)

# --- 4️⃣ Afficher les différences ---
# left_only : présent dans Spark, right_only : présent dans Pandas, both : présent dans les deux
print(comparison.head(20))


NameError: name 'final_summary_df' is not defined

In [31]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, explode, sum as spark_sum, count as spark_count, countDistinct, round as spark_round
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, BooleanType
import os

# --- 1. Créer la SparkSession ---
spark = SparkSession.builder \
    .appName("DailySummaryDebug") \
    .master("local[*]") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

# --- 2. Charger les fichiers ---
base_path = os.path.join('data-68ed', 'data', 'input')

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("is_active", BooleanType(), True)
])
customers_df = spark.read.csv(os.path.join(base_path, 'customers.csv'), header=True, schema=customers_schema)

refunds_schema = StructType([
    StructField("refund_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("reason", StringType(), True),
    StructField("created_at", StringType(), True)
])
refunds_df = spark.read.csv(os.path.join(base_path, 'refunds.csv'), header=True, schema=refunds_schema)

item_schema = StructType([
    StructField("qty", IntegerType(), True),
    StructField("sku", StringType(), True),
    StructField("unit_price", DoubleType(), True)
])
orders_schema = StructType([
    StructField("channel", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("items", ArrayType(item_schema), True),
    StructField("order_id", StringType(), True),
    StructField("payment_status", StringType(), True)
])
orders_df = spark.read.json(os.path.join(base_path, 'orders_2025-03-01.json'), schema=orders_schema, multiLine=True)

# --- 3. Nettoyage des données ---
# Clients actifs
active_customers_df = customers_df.filter(col("is_active") == True).select("customer_id", "city")

# Commandes payées uniquement
paid_orders_df = orders_df.filter(col("payment_status") == "paid")

# Exploser les articles et ne garder que les unit_price > 0
orders_exploded_df = paid_orders_df.withColumn("item", explode(col("items"))) \
    .filter(col("item.unit_price") > 0) \
    .withColumn("items_sold", col("item.qty")) \
    .withColumn("total_amount", col("item.qty") * col("item.unit_price"))

# Agg par commande pour avoir 1 ligne par order_id
orders_base_df = orders_exploded_df.groupBy("order_id", "customer_id", "channel", "created_at", "payment_status") \
    .agg(
        spark_sum("items_sold").alias("items_sold"),
        spark_sum("total_amount").alias("total_amount")
    )

# Dédupliquer sur order_id (sécurité)
orders_base_df = orders_base_df.dropDuplicates(["order_id"])

# Jointure avec clients actifs pour ne garder que les commandes valides
orders_clean_df = orders_base_df.join(active_customers_df, on="customer_id", how="inner")

# --- 4. Agrégation quotidienne ---
orders_clean_df = orders_clean_df.withColumn("date", to_date(col("created_at")))
refunds_df = refunds_df.withColumn("date", to_date(col("created_at")))

# Agrégations
sales_agg_df = orders_clean_df.groupBy("date", "city", "channel").agg(
    spark_count("order_id").alias("orders_count"),
    countDistinct("customer_id").alias("unique_customers"),
    spark_sum("items_sold").alias("items_sold"),
    spark_sum("total_amount").alias("gross_revenue_eur")
)

# Refunds
refunds_enriched_df = refunds_df.join(
    orders_clean_df.select("order_id", "city", "channel"),
    on="order_id",
    how="left"
)
refunds_agg_df = refunds_enriched_df.groupBy("date", "city", "channel").agg(
    spark_sum("amount").alias("refunds_eur")
)

# --- 5. Combinaison et calcul des revenus nets ---
final_summary_df = sales_agg_df.join(refunds_agg_df, ["date", "city", "channel"], "left") \
    .na.fill(0, ["refunds_eur"]) \
    .withColumn("net_revenue_eur", col("gross_revenue_eur") + col("refunds_eur")) \
    .withColumn("gross_revenue_eur", spark_round("gross_revenue_eur", 2)) \
    .withColumn("refunds_eur", spark_round("refunds_eur", 2)) \
    .withColumn("net_revenue_eur", spark_round("net_revenue_eur", 2))

# --- 6. Affichage pour vérification ---
final_summary_df.show(50, truncate=False)


+----+----+-------+------------+----------------+----------+-----------------+-----------+---------------+
|date|city|channel|orders_count|unique_customers|items_sold|gross_revenue_eur|refunds_eur|net_revenue_eur|
+----+----+-------+------------+----------------+----------+-----------------+-----------+---------------+
+----+----+-------+------------+----------------+----------+-----------------+-----------+---------------+



25/11/20 16:56:20 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: customer_id, email, city
 Schema: customer_id, city, is_active
Expected: city but found: email
CSV file: file:///workspace/data-68ed/data/input/customers.csv
